[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rsasaki0109/slam-handbook-python/blob/main/part1_foundations/ch02_state_representations/01_rotations_and_poses.ipynb)

# Chapter 2: 回転とポーズの表現 — Lie群入門

**SLAM Handbook Section 2.1** の内容をPythonで実装しながら学びます。

## このNotebookで学ぶこと

1. **SO(2)** — 2D回転行列、角度パラメータ
2. **SO(3)** — 3D回転行列、指数写像・対数写像
3. **SE(2), SE(3)** — ポーズ（回転+並進）の表現
4. **Lie群とLie代数** — wedge/vee, Exp/Log の関係
5. **マニフォールド上の最適化** — $\oplus / \ominus$ 演算子、接空間での更新

### なぜ重要か？
Ch01では回転角を単なるスカラーとして扱いましたが、3Dの回転は**マニフォールド**上の量です。
回転行列同士を足し算すると回転行列でなくなる → Lie群の枠組みが必要。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm, logm

np.set_printoptions(precision=4, suppress=True)
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 8)
plt.rcParams['font.size'] = 12

# 3Dプロット用（バージョン互換）
try:
    from mpl_toolkits.mplot3d import Axes3D
except ImportError:
    pass  # matplotlib >= 3.9 では自動的に利用可能

## 2.1 SO(2): 2D回転群

2D回転行列 $\mathbf{R} \in \text{SO}(2)$ は1つのパラメータ $\theta$ で表されます（SLAM Handbook Eq. 2.1）:

$$\mathbf{R} = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}$$

**性質**: $\mathbf{R}^\top\mathbf{R} = \mathbf{I}$, $\det(\mathbf{R}) = 1$

### Lie代数 so(2)

Lie代数の要素は **歪対称行列**（SLAM Handbook Eq. 2.5）:

$$\theta^\wedge = \begin{bmatrix} 0 & -\theta \\ \theta & 0 \end{bmatrix} \in \text{so}(2)$$

指数写像: $\mathbf{R} = \text{Exp}(\theta) = \exp(\theta^\wedge)$

In [ ]:
# =============================================================
# SO(2): 2D回転の実装
# =============================================================

class SO2:
    """SO(2) — 2D回転群"""
    
    @staticmethod
    def Exp(theta):
        """Lie代数 → Lie群 (指数写像)"""
        c, s = np.cos(theta), np.sin(theta)
        return np.array([[c, -s], [s, c]])
    
    @staticmethod
    def Log(R):
        """Lie群 → Lie代数 (対数写像)"""
        return np.arctan2(R[1, 0], R[0, 0])
    
    @staticmethod
    def wedge(theta):
        """スカラー → 歪対称行列 (∧ 演算子)"""
        return np.array([[0, -theta], [theta, 0]])
    
    @staticmethod
    def vee(Omega):
        """歪対称行列 → スカラー (∨ 演算子)"""
        return Omega[1, 0]

# テスト
theta = np.pi / 4  # 45°
R = SO2.Exp(theta)
print(f"θ = {np.degrees(theta):.0f}°")
print(f"R = Exp(θ) =\n{R}")
print(f"R^T R = {R.T @ R}  (= I ✓)")
print(f"det(R) = {np.linalg.det(R):.4f}  (= 1 ✓)")
print(f"Log(R) = {np.degrees(SO2.Log(R)):.0f}°  (= θ ✓)")

# 指数写像の検証: exp(θ^∧) = R
R_expm = expm(SO2.wedge(theta))
print(f"\nexp(θ^∧) =\n{R_expm}")
print(f"Exp(θ) と一致: {np.allclose(R, R_expm)} ✓")

In [ ]:
# =============================================================
# SO(2)の可視化: 回転行列がベクトルをどう変換するか
# =============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

angles = [30, 90, 150]
v = np.array([1.0, 0.0])  # 単位ベクトル

for ax, deg in zip(axes, angles):
    theta = np.radians(deg)
    R = SO2.Exp(theta)
    v_rot = R @ v
    
    # 原点と回転前後のベクトル
    ax.arrow(0, 0, v[0]*0.9, v[1]*0.9, head_width=0.06, head_length=0.05,
             fc='blue', ec='blue', linewidth=2)
    ax.arrow(0, 0, v_rot[0]*0.9, v_rot[1]*0.9, head_width=0.06, head_length=0.05,
             fc='red', ec='red', linewidth=2)
    
    # 回転の弧
    arc_angles = np.linspace(0, theta, 30)
    arc_x = 0.4 * np.cos(arc_angles)
    arc_y = 0.4 * np.sin(arc_angles)
    ax.plot(arc_x, arc_y, 'g-', linewidth=2)
    
    # 単位円
    circle = plt.Circle((0, 0), 1, fill=False, linestyle='--', alpha=0.3)
    ax.add_patch(circle)
    
    ax.set_xlim(-1.3, 1.3)
    ax.set_ylim(-1.3, 1.3)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.set_title(f'SO(2): θ = {deg}°', fontsize=13, fontweight='bold')
    ax.legend(['rotation arc'], loc='upper left')

plt.tight_layout()
plt.show()

## 2.2 SO(3): 3D回転群

3D回転行列 $\mathbf{R} \in \text{SO}(3)$ は3自由度（roll, pitch, yaw）を持ちます。

### Lie代数 so(3)

回転ベクトル $\boldsymbol{\theta} = [\theta_1, \theta_2, \theta_3]^\top \in \mathbb{R}^3$ に対して（SLAM Handbook Eq. 2.6b）：

$$\boldsymbol{\theta}^\wedge = \begin{bmatrix} 0 & -\theta_3 & \theta_2 \\ \theta_3 & 0 & -\theta_1 \\ -\theta_2 & \theta_1 & 0 \end{bmatrix} \in \text{so}(3)$$

### 指数写像（Rodrigues公式）

$$\mathbf{R} = \text{Exp}(\boldsymbol{\theta}) = \mathbf{I} + \frac{\sin\phi}{\phi} \boldsymbol{\theta}^\wedge + \frac{1 - \cos\phi}{\phi^2} (\boldsymbol{\theta}^\wedge)^2$$

ここで $\phi = \|\boldsymbol{\theta}\|$ は回転角度。

In [ ]:
# =============================================================
# SO(3): 3D回転の実装
# =============================================================

class SO3:
    """SO(3) — 3D回転群"""
    
    @staticmethod
    def wedge(theta):
        """R^3 → so(3) 歪対称行列 (∧ 演算子)"""
        return np.array([
            [0, -theta[2], theta[1]],
            [theta[2], 0, -theta[0]],
            [-theta[1], theta[0], 0]
        ])
    
    @staticmethod
    def vee(Omega):
        """so(3) → R^3 (∨ 演算子)"""
        return np.array([Omega[2, 1], Omega[0, 2], Omega[1, 0]])
    
    @staticmethod
    def Exp(theta):
        """R^3 → SO(3) 指数写像 (Rodrigues公式)"""
        theta = np.asarray(theta, dtype=float)
        phi = np.linalg.norm(theta)
        if phi < 1e-10:
            return np.eye(3) + SO3.wedge(theta)
        
        a = theta / phi  # 回転軸（単位ベクトル）
        a_wedge = SO3.wedge(a)
        return np.eye(3) + np.sin(phi) * a_wedge + (1 - np.cos(phi)) * (a_wedge @ a_wedge)
    
    @staticmethod
    def Log(R):
        """SO(3) → R^3 対数写像"""
        cos_phi = (np.trace(R) - 1) / 2
        cos_phi = np.clip(cos_phi, -1, 1)
        phi = np.arccos(cos_phi)
        
        if phi < 1e-10:
            return SO3.vee(R - np.eye(3))
        
        log_R = phi / (2 * np.sin(phi)) * (R - R.T)
        return SO3.vee(log_R)

# テスト: z軸周りに90°回転
theta = np.array([0, 0, np.pi/2])
R = SO3.Exp(theta)
theta_recovered = SO3.Log(R)

print(f"θ = {np.degrees(theta)}°")
print(f"R = Exp(θ) =\n{R}")
print(f"R^T R =\n{R.T @ R}")
print(f"det(R) = {np.linalg.det(R):.4f}")
print(f"Log(R) = {np.degrees(theta_recovered)}°")

# scipy.linalg.expm との比較
R_expm = expm(SO3.wedge(theta))
print(f"\nexpm(θ^∧) との差: {np.linalg.norm(R - R_expm):.2e} ✓")

In [ ]:
# =============================================================
# SO(3)の3D可視化: 座標軸の回転
# =============================================================

def plot_frame(ax, R, t=np.zeros(3), scale=1.0, label='', alpha=1.0):
    """3D座標フレームを描画"""
    colors = ['r', 'g', 'b']
    labels_axis = ['x', 'y', 'z']
    for i in range(3):
        direction = R[:, i] * scale
        ax.quiver(t[0], t[1], t[2], direction[0], direction[1], direction[2],
                  color=colors[i], arrow_length_ratio=0.1, linewidth=2, alpha=alpha)
    if label:
        ax.text(t[0], t[1], t[2] - 0.2, label, fontsize=10, fontweight='bold')

fig = plt.figure(figsize=(15, 5))

rotations = [
    ('Identity', np.eye(3)),
    ('Rx(45°)', SO3.Exp([np.pi/4, 0, 0])),
    ('Ry(60°)', SO3.Exp([0, np.pi/3, 0])),
    ('Rz(90°)', SO3.Exp([0, 0, np.pi/2])),
]

for idx, (name, R) in enumerate(rotations):
    ax = fig.add_subplot(1, 4, idx+1, projection='3d')
    plot_frame(ax, np.eye(3), scale=0.5, alpha=0.3)  # 元のフレーム（薄く）
    plot_frame(ax, R, label=name)  # 回転後のフレーム
    ax.set_xlim([-1, 1]); ax.set_ylim([-1, 1]); ax.set_zlim([-1, 1])
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
    ax.set_title(name, fontweight='bold')

plt.tight_layout()
plt.show()

## 2.3 SE(3): 3Dポーズ（回転+並進）

ポーズは回転と並進の組み合わせで、**同次変換行列** $\mathbf{T} \in \text{SE}(3)$ で表します（SLAM Handbook Eq. 2.2）:

$$\mathbf{T} = \begin{bmatrix} \mathbf{R} & \mathbf{t} \\ \mathbf{0} & 1 \end{bmatrix} \in \text{SE}(3)$$

### Lie代数 se(3)

6次元ベクトル $\boldsymbol{\xi} = [\boldsymbol{\rho}^\top, \boldsymbol{\theta}^\top]^\top \in \mathbb{R}^6$ で表現（SLAM Handbook Eq. 2.7b）:

$$\boldsymbol{\xi}^\wedge = \begin{bmatrix} \boldsymbol{\theta}^\wedge & \boldsymbol{\rho} \\ \mathbf{0} & 0 \end{bmatrix} \in \text{se}(3)$$

### $\oplus$ と $\ominus$ 演算子

マニフォールド上での「加算」と「差分」（SLAM Handbook Eq. 2.24-2.25）:

$$\mathbf{T} = \mathbf{T}^0 \oplus \boldsymbol{\xi} = \mathbf{T}^0 \, \text{Exp}(\boldsymbol{\xi})$$
$$\boldsymbol{\xi} = \mathbf{T} \ominus \mathbf{T}^0 = \text{Log}(\mathbf{T}^{0^{-1}} \mathbf{T})$$

In [ ]:
# =============================================================
# SE(3): 3Dポーズの実装
# =============================================================

class SE3:
    """SE(3) — 3Dポーズ群"""
    
    @staticmethod
    def from_Rt(R, t):
        """回転行列と並進ベクトルからSE(3)行列を作る"""
        T = np.eye(4)
        T[:3, :3] = R
        T[:3, 3] = t
        return T
    
    @staticmethod
    def R(T):
        """SE(3)行列から回転部分を取得"""
        return T[:3, :3]
    
    @staticmethod
    def t(T):
        """SE(3)行列から並進部分を取得"""
        return T[:3, 3]
    
    @staticmethod
    def inv(T):
        """SE(3)の逆行列: T^{-1} = [R^T, -R^T t; 0, 1]"""
        R = T[:3, :3]
        t = T[:3, 3]
        T_inv = np.eye(4)
        T_inv[:3, :3] = R.T
        T_inv[:3, 3] = -R.T @ t
        return T_inv
    
    @staticmethod
    def wedge(xi):
        """R^6 → se(3) (∧ 演算子)
        xi = [rho(3), theta(3)]
        """
        rho, theta = xi[:3], xi[3:]
        Xi = np.zeros((4, 4))
        Xi[:3, :3] = SO3.wedge(theta)
        Xi[:3, 3] = rho
        return Xi
    
    @staticmethod
    def vee(Xi):
        """se(3) → R^6 (∨ 演算子)"""
        rho = Xi[:3, 3]
        theta = SO3.vee(Xi[:3, :3])
        return np.concatenate([rho, theta])
    
    @staticmethod
    def Exp(xi):
        """R^6 → SE(3) 指数写像"""
        return expm(SE3.wedge(xi))
    
    @staticmethod
    def Log(T):
        """SE(3) → R^6 対数写像"""
        return SE3.vee(logm(T))
    
    @staticmethod
    def oplus(T, xi):
        """⊕ 演算子: T ⊕ ξ = T · Exp(ξ)"""
        return T @ SE3.Exp(xi)
    
    @staticmethod
    def ominus(T1, T2):
        """⊖ 演算子: T1 ⊖ T2 = Log(T2^{-1} · T1)"""
        return SE3.Log(SE3.inv(T2) @ T1)

# テスト
R = SO3.Exp([0, 0, np.pi/4])   # z軸45°回転
t = np.array([1.0, 2.0, 0.5])   # 並進
T = SE3.from_Rt(R, t)

print("=== SE(3) ポーズ ===")
print(f"T =\n{T}")
print(f"\nT^{{-1}} · T =\n{SE3.inv(T) @ T}")

# ⊕ と ⊖ のテスト
xi = np.array([0.1, 0.2, 0.0, 0.0, 0.0, np.pi/6])  # 小さな摂動
T2 = SE3.oplus(T, xi)
xi_recovered = SE3.ominus(T2, T)
print(f"\nξ (元):      {xi}")
print(f"T2⊖T (復元): {xi_recovered}")
print(f"一致: {np.allclose(xi, xi_recovered)} ✓")

## 2.4 マニフォールド上の最適化

### なぜ普通の加算ではダメか

回転行列に小さなベクトルを足しても、結果は回転行列にならない：
$\mathbf{R} + \boldsymbol{\delta} \notin \text{SO}(3)$

### 正しい方法: Lie代数で摂動 → Exp写像でマニフォールドに戻す

$$\mathbf{T}^{t+1} = \mathbf{T}^t \oplus \boldsymbol{\xi}^* = \mathbf{T}^t \, \text{Exp}(\boldsymbol{\xi}^*)$$

最適化は $\boldsymbol{\xi} \in \mathbb{R}^6$（接空間）で行い、更新時に $\text{Exp}$ でSE(3)に戻す。

これが **Ch01のGauss-Newton法** との違い：状態更新が $\mathbf{x} \leftarrow \mathbf{x} + \boldsymbol{\delta}$ ではなく $\mathbf{T} \leftarrow \mathbf{T} \cdot \text{Exp}(\boldsymbol{\xi})$ になる。

In [ ]:
# =============================================================
# デモ: 加算 vs ⊕演算子
# =============================================================

R_true = SO3.Exp([0, 0, np.pi/3])  # 60°回転

# 悪い方法: 回転行列に直接加算
delta_bad = np.array([[0, -0.1, 0], [0.1, 0, 0], [0, 0, 0]])
R_bad = R_true + delta_bad

# 良い方法: Lie代数で摂動
delta_good = np.array([0, 0, 0.1])  # z軸周りに小さな追加回転
R_good = R_true @ SO3.Exp(delta_good)

print("=== 加算 vs ⊕ の比較 ===")
print(f"\n直接加算 R + δ:")
print(f"  R^T R = {R_bad.T @ R_bad}")
print(f"  det(R) = {np.linalg.det(R_bad):.4f}")
print(f"  → SO(3)の制約を満たさない!")

print(f"\n⊕ 演算子 R · Exp(δ):")
print(f"  R^T R = {R_good.T @ R_good}")
print(f"  det(R) = {np.linalg.det(R_good):.4f}")
print(f"  → SO(3)の制約を満たす ✓")

In [ ]:
# =============================================================
# SE(3)ポーズ列の3D可視化
# =============================================================

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# ロボットが移動するポーズ列を生成
n_poses = 8
poses = [np.eye(4)]

for i in range(n_poses - 1):
    # 各ステップで少し前進 + 少し回転
    xi = np.array([0.5, 0.0, 0.1,    # rho (前進)
                   0.0, 0.0, np.pi/6])  # theta (z軸回転)
    poses.append(SE3.oplus(poses[-1], xi))

# 描画
for i, T in enumerate(poses):
    R = SE3.R(T)
    t = SE3.t(T)
    plot_frame(ax, R, t, scale=0.3)
    if i > 0:
        t_prev = SE3.t(poses[i-1])
        ax.plot([t_prev[0], t[0]], [t_prev[1], t[1]], [t_prev[2], t[2]],
                'k--', alpha=0.5)

ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_title('SE(3) Pose Chain: ⊕ 演算子で逐次更新', fontsize=14, fontweight='bold')

# 視点調整
ax.view_init(elev=30, azim=-60)
plt.tight_layout()
plt.show()

print("→ 各ポーズは T_{i+1} = T_i ⊕ ξ で計算")
print("→ 回転行列の制約 (R^T R = I, det=1) が常に保たれる")

## 2.5 Exp/Logの関係図

SLAM Handbook Fig. 2.2 の内容をまとめます:

```
  ℝⁿ  ──(∧)──→  so(n)  ──exp──→  SO(n)
   │               │                │
   │ パラメータ空間  │   Lie代数       │  Lie群
   │ (ベクトル)      │  (接空間)       │  (マニフォールド)
   │               │                │
  ℝⁿ  ←──(∨)───  so(n)  ←─log──  SO(n)
```

- `Exp(θ) = exp(θ^∧)`: ベクトル → 回転行列
- `Log(R) = (log(R))^∨`: 回転行列 → ベクトル
- `∧` (wedge): ベクトル → 歪対称行列
- `∨` (vee): 歪対称行列 → ベクトル

## 2.6 演習問題

### 演習1: 回転の合成
2つの回転 $\mathbf{R}_1 = \text{Exp}([0, 0, \pi/4])$, $\mathbf{R}_2 = \text{Exp}([\pi/6, 0, 0])$ を合成し、$\text{Log}(\mathbf{R}_1 \mathbf{R}_2)$ を計算してみましょう。$\text{Log}(\mathbf{R}_1) + \text{Log}(\mathbf{R}_2)$ と一致しますか？なぜ？

### 演習2: Adjoint表現
SE(3)のAdjoint行列 $\text{Ad}(\mathbf{T})$（SLAM Handbook Eq. 2.30）を実装し、$\mathbf{T}\text{Exp}(\boldsymbol{\xi}) = \text{Exp}(\text{Ad}(\mathbf{T})\boldsymbol{\xi})\mathbf{T}$ が成り立つことを確認してください。

### 演習3: SE(3)上のPose Graph最適化
Ch01のGauss-Newton法を修正して、状態更新を $\mathbf{T} \leftarrow \mathbf{T} \cdot \text{Exp}(\boldsymbol{\xi}^*)$ に変えた3D Pose Graph SLAMを実装してみましょう。

## まとめ

| 群 | 自由度 | 要素 | Lie代数 | 用途 |
|---|---|---|---|---|
| SO(2) | 1 | 2×2回転行列 | so(2) | 2D回転 |
| SO(3) | 3 | 3×3回転行列 | so(3) | 3D回転 |
| SE(2) | 3 | 3×3同次行列 | se(2) | 2Dポーズ |
| SE(3) | 6 | 4×4同次行列 | se(3) | 3Dポーズ |

- **Exp写像**: Lie代数（接空間）→ Lie群（マニフォールド）
- **Log写像**: Lie群 → Lie代数
- **⊕/⊖演算子**: マニフォールド上での更新と差分
- **最適化**: 接空間 $\boldsymbol{\xi} \in \mathbb{R}^n$ で線形化し、$\text{Exp}$ で戻す

### 次のNotebook
→ `02_lie_groups.ipynb`: Lie群のヤコビアン、不確かさの表現、SE(3)上の最適化